In [ ]:
import sys
import os
from pathlib import Path
from natsort import natsorted

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc


import warnings
warnings.filterwarnings('ignore')

: 

In [ ]:

from pathlib import Path
import os

home_dir = Path.home()

# QC kaynağı: script01'in cellbin çıktıları
src_dir = home_dir / 'ext_hd_sammy' / 'projects' / 'out' / 'out_stomics' / 'script01_output' 

# QC çıktıları buraya kaydedilecek
dst_folder = home_dir / 'ext_hd_sammy' / 'projects' / 'out' / 'out_stomics' / 'script02_output'
dst_folder.mkdir(parents=True, exist_ok=True)

# sadece bu Chip ID'leri QC yap
chip_whitelist = {"C04143D3","B04101E6","C04143G2","B04101A3","A04100A6","C04139D3"}


### HPC
#src_dir = home_dir / 'scratch' / 'projects' / 'sammy' / 'out' / 'out_stomics' / 'script01_output'
#dst_folder = home_dir / 'scratch' / 'projects' / 'sammy' / 'out' / 'out_stomics' / 'script02_output'



os.makedirs(dst_folder, exist_ok=True)



In [ ]:
folders = os.listdir(src_dir)
adata_foldernames = natsorted(folders)
adata_foldernames

In [ ]:
import subprocess

median_ncounts = []
median_ngenes = []
median_ncounts_q99 = []
median_ngenes_q99 = []
median_ncounts_q01 = []
median_ngenes_q01 = []


### plot the distribution of ncounts and ngenes
#h5ad_files = [f for f in os.listdir(src_dir) if f.endswith('.h5ad') and f.split('_')[0] in chip_whitelist]
#h5ad_files = natsorted(h5ad_files)

for folder in adata_foldernames:
    src_subdir = src_dir / folder
    chips = os.listdir(src_subdir)
    
    dst_subdir = dst_folder / folder
    os.makedirs(dst_subdir, exist_ok=True)

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    for chip_id in chip_whitelist:
        adata_filepath = src_subdir / f'{chip_id}_{folder}.h5ad'
        
    
    
    
        metadata_df = pd.DataFrame(columns=['sample_id', 'median_ncounts', 'median_ngenes',
                                        'median_ncounts_q99', 'median_ngenes_q99',
                                        'median_ncounts_q01', 'median_ngenes_q01'])



        adata = sc.read_h5ad(adata_filepath)
        adata.uns['sample_id'] = chip_id
        adata.obs['sample_id'] = chip_id
        adata.obs['sample_id'] = adata.obs['sample_id'].astype('category')
        
        
        ncounts = adata.X.sum(axis=1)
        adata.obs['ncounts'] = ncounts

        ngenes = (adata.X > 0).sum(axis=1)
        adata.obs['ngenes'] = ngenes


        median_ncounts.append(adata.obs['ncounts'].median())
        median_ngenes.append(adata.obs['ngenes'].median())
        median_ncounts_q99.append(adata.obs['ncounts'].quantile(0.99))
        median_ngenes_q99.append(adata.obs['ngenes'].quantile(0.99))
        median_ncounts_q01.append(adata.obs['ncounts'].quantile(0.01))
        median_ngenes_q01.append(adata.obs['ngenes'].quantile(0.01))


        sns.histplot(adata.obs['ncounts'], ax=axes[0], kde=True, color='blue', label=chip_id)
        axes[0].set_title(f'Distribution of ncounts for {chip_id}')
        axes[0].set_xlabel('ncounts')
        ### set vertical line at median
        axes[0].axvline(adata.obs['ncounts'].median(), color='red', linestyle='--', label='Median ncounts')
        axes[0].axvline(adata.obs['ncounts'].quantile(0.99), color='green', linestyle='--', label='99th percentile ncounts')
        axes[0].axvline(adata.obs['ncounts'].quantile(0.01), color='orange', linestyle='--', label='1st percentile ncounts')
        axes[0].set_xlim(0, 1000)

        axes[0].legend()


        sns.histplot(adata.obs['ngenes'], ax=axes[1], kde=True, color='orange', label=chip_id)
        axes[1].set_title(f'Distribution of ngenes for {chip_id}')
        axes[1].set_xlabel('ngenes')
        ### set vertical line at median
        axes[1].axvline(adata.obs['ngenes'].median(), color='red', linestyle='--', label='Median ngenes')
        axes[1].axvline(adata.obs['ngenes'].quantile(0.99), color='green', linestyle='--', label='99th percentile ngenes')
        axes[1].axvline(adata.obs['ngenes'].quantile(0.01), color='orange', linestyle='--', label='1st percentile ngenes')
        axes[1].set_xlim(0, 1000)
        axes[1].legend()
        


        adata.raw = adata
        
        sc.pp.filter_cells(adata, min_genes=adata.obs['ngenes'].quantile(0.01))
        sc.pp.filter_cells(adata, max_counts=adata.obs['ncounts'].quantile(0.99))
        sc.pp.filter_cells(adata, min_counts=adata.obs['ncounts'].quantile(0.01))

        
        '''
        try:
            sc.pp.normalize_total(adata, target_sum=1e4)
            #sc.experimental.pp.normalize_pearson_residuals(adata)
            sc.pp.log1p(adata)
            sc.pp.scale(adata, max_value=10)
            sc.pp.highly_variable_genes(adata, n_top_genes=300)
            sc.pp.pca(adata, n_comps=21, svd_solver='arpack', use_highly_variable=True)
            sc.pp.neighbors(adata, n_neighbors=15, n_pcs=21)
            sc.tl.umap(adata, min_dist=0.3, spread=1.0, n_components=2)
            sc.tl.leiden(adata, resolution=0.1)
            sc.tl.rank_genes_groups(adata, groupby='leiden', method='t-test', n_genes=adata.shape[1])

        except Exception as e:
            print(f"Error processing {chip_id}: {e}")
            continue
        
    
    
        metadata_file_path = os.path.join(dst_subdir / 'metadata_transcripts.csv')
        if not os.path.exists(metadata_file_path):
            subprocess.run(['touch', metadata_file_path])
            
       '''
       
        with open(dst_subdir / 'metadata_transcripts.csv', 'a') as f:
            f.write(f"Sample ID: {chip_id}\n")
            f.write(f"Median ncounts: {adata.obs['ncounts'].median()}\n")
            f.write(f"Median ngenes: {adata.obs['ngenes'].median()}\n")
            f.write(f"99th percentile ncounts: {adata.obs['ncounts'].quantile(0.99)}\n")
            f.write(f"99th percentile ngenes: {adata.obs['ngenes'].quantile(0.99)}\n")
            f.write(f"1st percentile ncounts: {adata.obs['ncounts'].quantile(0.01)}\n")
            f.write(f"1st percentile ngenes: {adata.obs['ngenes'].quantile(0.01)}\n")

    
        metadata = pd.DataFrame({
            'sample_id': chip_id.split('.')[0],
            'median_ncounts': adata.obs['ncounts'].median(),
            'median_ngenes': adata.obs['ngenes'].median(),
            'median_ncounts_q99': adata.obs['ncounts'].quantile(0.99),
            'median_ngenes_q99': adata.obs['ngenes'].quantile(0.99),
            'median_ncounts_q01': adata.obs['ncounts'].quantile(0.01),
            'median_ngenes_q01': adata.obs['ngenes'].quantile(0.01)
        }, index=[0])

        
        metadata_df = pd.concat([metadata_df, metadata], ignore_index=True, axis=0)
        # Save metadata to a file
        print(f"Saving processed data for {chip_id}_{folder}...")

        #Save metadata to a file
        try:
            adata.write_h5ad(dst_subdir / f'{chip_id}_{folder}.h5ad')
        except Exception as e:
            print(f"Error occurred while saving {chip_id}_{folder}: {e}")
            ### delete the file if it exists
            if (dst_subdir / f'{chip_id}_{folder}.h5ad').exists():
                (dst_subdir / f'{chip_id}_{folder}.h5ad').unlink()


    plt.tight_layout()
    plt.savefig(dst_subdir / 'distribution_ncounts_ngenes.png', bbox_inches='tight')
    plt.close(fig)

    # Save the metadata DataFrame to a CSV file
#metadata_df.to_csv(dst_folder / f'{chip_id}_metadata_transcripts.csv', index=False)
#print("Metadata saved to metadata_transcripts.csv")
#plt.savefig(dst_subdir / 'distribution_ncounts_ngenes.png', bbox_inches='tight')


In [ ]:
adata